# Full Lecture Summary
please note: contents of this file might be incorrect, for questions contact gtenzing@student.ethz.ch 

In [ ]:
# necessary libraries 
import numpy as np
from numpy.linalg    import norm, solve, matrix_rank

from scipy.special   import roots_jacobi, roots_legendre
from scipy.integrate import quad, solve_ivp
from scipy.optimize  import fsolve
from scipy.linalg    import expm, solve, qr, svd, lstsq, pinv

import matplotlib.pyplot as plt
import sympy as sp

: 

## Chapter 2: Numerische Quadratur

In [ ]:
def quadrature_rule(f, a, b, N):
    x, h = np.linspace(a, b, N + 1, retstep=True)
    xm   = 0.5* (x[1:] + x[:-1]) 
    
    I_midp = h       *                                        sum(f(xm))             # Page: 14
    I_trap = h / 2.0 * (f(x[0]) + 2.0 * sum(f(x[1:-1]))                  + f(x[-1])) # Page: 16
    I_simp = h / 6.0 * (f(x[0]) + 2.0 * sum(f(x[1:-1])) + 4.0*sum(f(xm)) + f(x[-1])) # Page: 16
    I_ga2p = 0.5 * h * np.sum(f(xm - h / 2 * np.sqrt(3)) + f(xm +  h / 2 * np.sqrt(3)) )
    
    x13  = x[:-1] + 1/3 * h
    x23  = x[:-1] + 2/3 * h 
    I_sm38 = h / 8.0 * (f(x[0]) + 3.0 * sum(f(x13) + f(x23)) + 2.0 * sum(f(x[1:-1])) + f(x[-1]))
    
    c_cotes, w_cotes = compute_weights(a, b, s = 5)
    c_radau, w_radau = radau(s = 5, fixed = "l")
    c_lobat, w_lobat = lobatto(s = 5)
    I_cotes = sum(w_cotes * f(c_cotes))
    I_radau = sum(w_radau * f(c_radau))
    I_lobat = sum(w_lobat * f(c_lobat))
    
    return [I_midp, I_trap, I_simp, I_ga2p, I_cotes, I_radau, I_lobat]

def radau(s, fixed): # Page: 23 
    ws = 2 / (s**2)
    if fixed == 'r':
        nodes, w = roots_jacobi(s - 1, alpha = 1, beta = 0)  # Jacobi nodes and weights 
        c = np.hstack([nodes, 1])                        # Radau nodes
        b = np.hstack([(w / (1 - nodes)) , ws])          # Radau weights (1-t)**1 * (1+t)**0
    elif fixed == 'l':
        nodes, w = roots_jacobi(s - 1, alpha = 0, beta = 1)  # Jacobi nodes and weights
        c = np.hstack([-1, nodes])                       # Radau nodes
        b = np.hstack([ws, (w / (1 + nodes))])           # Radau weights (1-t)**0 * (1+t)**1
    return c, b

def lobatto(s): # Page: 23 
    inside_nodes, w = roots_jacobi(s - 2, alpha=1, beta=1) # Jacobi nodes and weights
    c = np.hstack([-1, inside_nodes, 1])                   # Lobatto nodes
    w1 = ws = 2 / ((s - 1) * s)                            # Lobatto weights at -1 and 1                         
    b = np.hstack([w1, w / (1 - inside_nodes**2), ws])     # Lobatto weights [-1,1] (1-t)**1 * (1+t)**1
    return c, b


def transform_quadrature_interval(nodes, weights, a, b):
    transformed_nodes   = 0.5 * (b - a) * nodes + 0.5 * (a + b)
    transformed_weights = 0.5 * (b - a) * weights
    return transformed_nodes, transformed_weights

def compute_weights(a, b, s, x=None): # Page: not 10
    c   = np.linspace(a, b, s)                            # equidistant nodes 
    c   = x                                               # if nodes are given

    # V = np.array([c**i for i in range(s)])                 
    V   = np.vander(c, N=s, increasing=True).T            # Vandermonde matrix of the nodes c
    rhs = np.array([(b**(k + 1) - a**(k + 1)) / (k + 1)   # exact integral of monomials x^p from a to b
                    for k in range(s) ])
    
    w   = np.linalg.solve(V, rhs)                         # w = M^{-1} * rhs   
    return c, w

def compute_weights_and_nodes(): # if both nodes and weights are not given
    def equations(vars):
        x1, x2, w1, w2 = vars
        rhs = [1, 0, 1/3, 0] # [-1,1]
        # rhs = [1, 1/2, 1/3, 1/4] # if [a,b] = [0,1]
        return [w1 * x1**i + w2 * x2**i - rhs[i] for i in range(len(rhs))]
    
    return fsolve(equations, [1, 1, 1, 1])

def compute_errors_and_order(integrand, a, b, I_ref, quadrature): # Page: 17 
    n_evals = 2.0 ** np.arange(3, 10)

    errors = np.array([abs(I_ref - quadrature(integrand, a, b, n)) for n in n_evals]) 
    order  = - np.polyfit(np.log(n_evals), np.log(errors), deg=1)[0]

    plt.loglog(n_evals, errors,         label=quadrature.__name__)
    plt.loglog(n_evals, n_evals**order, label=r"$n^{-2.0}$")
    return errors, order

def compute_order_and_genauigkeitsgrad(method, n, tol = 1e-11): 
    a, b = -1, 1 # Interval for the quadrature rule
    c, w = method(n)
    max_degree = 2 * n
    for k in range(max_degree + 1):
        f_ref = lambda x: x ** k  
        I_exact  = (b**(k + 1) - a**(k + 1)) / (k + 1)  # = quad(f, a, b)[0] if [a,b] =! [-1,1]
        I_approx = np.sum(w * f_ref(c))
        if abs(I_approx - I_exact) > tol:
            return k, k-1 

def gauss(f, a, b, N):
    x, h = np.linspace(a, b, N + 1, retstep=True)
    [nodes, weights] = roots_legendre(5)

    I_gaus =    0.5 * (b - a) * weights @ f(0.5  * (b - a) * nodes + 0.5  * (a + b)) # Page: 28
    I_comp = np.sum([ 0.5 * h * weights @ f(xi + 0.5 * h * (nodes + 1))  for xi in x[:-1] ]) 
    return [I_gaus, I_comp]

## Chapter 3: Trigonometrische Interpolation

In [ ]:
def evaliptrig(f_samples, N_target): # Page 84     # Evaluate interpolating trigonometric polynomial at N points
    n_samples = len(f_samples)                     # Number of input samples (assumed periodic samples of a function)
    max_freq = n_samples // 2                      # Maximum frequency (Nyquist frequency) for the input samples

    c = np.fft.ifft(f_samples)                     # Compute Fourier coefficients (inverse FFT of sample values)
    # freq_indices_k = np.fft.fftshift(np.arange(- max_freq, max_freq)) for derivative
    # df_hat  = - 2j * np.pi * freq_indices_k * fourier_coeffs

    padded = np.zeros(N_target, dtype = complex)    # Initialize zero-padded array for upsampling (length N)
    padded[ : max_freq]            = c[ : max_freq] # Copy first half of Fourier coefficients into beginning of a
    padded[N_target - max_freq : ] = c[max_freq : ] # Copy second half into end of -||- (preserves symmetry / periodicity)
    
    f_interp = np.real(np.fft.fft(padded))          # Compute FFT of padded spectrum → evaluates interpolated values
    return f_interp                                   

def convtrig(f):     # Page 84                         # Use up to n = 128 points; power of 2 is ideal for FFT-based interpolation
    x_fine = np.linspace(0, 1, N_ref, endpoint=False)  # Reference grid of N points in [0,1)
    f_exact   = f(x_fine)                              # Evaluate f at the reference points (ground truth)
    
    N_ref = 2**15
    n_max = 2**7 + 1
    n_values = np.arange(2, n_max, 4)                  # List to store number of interpolation points
    max_errors  = []                                   # List to store L∞ errors 
    
    for n in n_values:                                      # Loop over number of interpolation points
        x_coarse  = np.linspace(0, 1, n, endpoint=False)    # Generate n equally spaced points on [0,1)
        f_samples = f(x_coarse)                             # Evaluate f at those interpolation points  
        f_interp  = evaliptrig(f_samples, N_ref)            # Evaluate trigonometric interpolant on fine grid

        # max_error = abs(f_interp - f_exact).max()         # Compute (L∞ norm) max pointwise error between interpolated and true values
        max_error = norm(f_interp - f_exact) / np.sqrt(N_ref)
        max_errors.append(max_error)                      # Store error in list

    # Optional plotting of interpolated functions 
    plt.plot(x_fine, f_exact)                             # Plot exact         function (fine resolution)
    plt.plot(x_fine, f_interp)                            # Plot interpolated  function (coarse resolution)  
    return n_values, max_errors                      # Return (n, error) pairs for plotting or analysis

# def plot_fourier_interpolation():
#     N_ref = 4096                                             # High-resolution number of evaluation points
#     n_samples = 128                                          # Low-resolution Number of sample points for interpolation

#     fine_gird   = np.linspace(0, 1, N_ref, endpoint=False)   # High-res grid over [0, 1] with N points (no endpoint)
#     x_coarse = np.linspace(0, 1, n_samples, endpoint=False)  # Generate n equally spaced points on [0,1)

#     f_exact = f(fine_gird)                                   # Evaluate exact step function on fine grid

#     f_samples = f(x_coarse)                                  # Evaluate f at those interpolation points
#     f_interp = evaliptrig(f_samples, N_ref)                  # Evaluate trigonometric interpolant on fine grid and take real part
        
#     plt.plot(fine_gird, f_exact)                             # Plot exact         function (fine resolution)
#     plt.plot(fine_gird, f_interp)                            # Plot interpolated  function (coarse resolution)                      

def plot_fourier_coeffs():
    n_samples = 2**10                                                # Number of points to sample the function (grid size)
    sample_points, h = np.linspace(0, 1, n_samples, endpoint=False, retstep=True)     # Uniform sampling points in [0, 1)

    f_samples                  = f(sample_points)                    # Evaluate function at those sample points
    fourier_coeffs             = np.fft.ifft(f_samples)              # Compute Fourier coefficients (via IFFT)
    fourier_coeffs_centered    = np.fft.fftshift(fourier_coeffs)     # Center Fourier coefficients
    fourier_magnitude_centered = abs(fourier_coeffs_centered)        # Magnitudes of Fourier coefficients 

    half_n_samples = n_samples / 2
    # freq_indices_centered_1 = np.arange(- half_n_samples, half_n_samples)   # Frequency index axis (centered around 0)
    freq_indices_centered_2 = np.fft.fftshift(np.fft.fftfreq(n_samples, d=h)) # h = 1.0/n_samples

    plt.semilogy(freq_indices_centered_2, fourier_magnitude_centered)  



## Chapter 4: Gewöhnliche Differentialgleichungen

In [ ]:
def pendulum_rhs(t, y):
    # y   = [winkel, winkel-geschw.]     = [phi, phi_dot]
    # rhs = [winkel-geschw. winkel-acc.] = [phi_dot, phi_dot_dot]
    rhs = np.array([y[1], -9.81 / 0.6 * np.sin(y[0])])
    return rhs

def integrate(method_step, rhs, y0, T, N): # Page: 191
    t, h = np.linspace(0.0, T, N + 1, retstep=True)

    y = np.empty((N + 1,) + y0.shape)
    y[0, :] = y0
    
    for i in range(N):
        y[i + 1, :] = method_step(rhs, y[i, :], t[i], h)

    return t, y

def compute_and_plot_exact_errors_rate(method_step):
    eval = 2 ** np.arange(4,15); T  = 10; N = 420; y0 = np.array([1.0, 0.0]) 
    rhs = lambda t, y: np.array([y[1], -9.81 / 0.6 * np.sin(y[0])]) # Example: Harmonic Oscillator

    # Compute: reference exact solution # Page 206
    ref_sol = solve_ivp(rhs, t_span=(0.0, T), y0=y0, method="RK45", atol=1e-16, rtol=1e-10) 
    t_exact = ref_sol.t; y_exact = ref_sol.y.T
    # Optional: t_eval = [10] get sol for t = 10 only

    # Plot: Method and Exact 
    t_approx, y_approx = integrate(method_step, rhs, y0, T, N)
    plt.plot(t_approx, y_approx[:, 0]) 
    plt.plot(t_exact , y_exact [:, 0]) 
    plt.show()

    # Compute and Plot: Errors and Rate 
    errors = [abs(y_exact[-1, 0] - integrate(method_step, rhs, y0, T, n)[1][-1, 0]) for n in eval]
    order   = - np.polyfit(np.log(eval), np.log(errors), deg=1) [0]
    plt.loglog(eval, errors)
    plt.loglog(eval, eval**order, label=f"conv order = {order}")

def compute_crit_N(method, rhs, y0, ye):
    f_err = lambda n: compute_error(rhs, y0, n, ye, method)[0]
    N_min = bisect(f_err, n_low = 1, n_high = 5000, fx_crit = 1e-4)

def method_step(rhs, y0, t0, dt):
    eE = y0 +       dt * rhs(t0, y0)
    eM = y0 +       dt * rhs(t0 + 0.5 * dt, y0 + 0.5 * dt * rhs(t0,y0))
    eT = y0 + 0.5 * dt *(rhs(t0, y0) + rhs(t0 + dt, y0 + dt * rhs(t0, y0)))
    iE = fsolve(lambda y1: y1 - (y0 + dt * rhs(t0 + dt, y1)),                    eE)
    iM = fsolve(lambda y1: y1 - (y0 + dt * rhs(t0 + 0.5 * dt, 0.5 * (y0 + y1))), eE)
    mod_iE =  y0 + dt * rhs(t0 + 0.5 * dt, fsolve(lambda y1: y1 - (y0 + 0.5 * dt * rhs(t0 + 0.5 * dt, y1)), y0))

def velocity_verlet_step(rhs, y0, t0, dt):
    y0 = y0.reshape((2, 1))  # y = [x, v]
    y1 = np.empty_like(y0)
    
    x0, v0 = y0[0, :], y0[1, :]
 
    y1[0, :] = x0 + v0 * dt + 0.5 * rhs(t0, x0) * dt**2                    
    y1[1, :] =           v0 + 0.5 * (rhs(t0, x0) + rhs(t0 + dt, y1[0, :])) * dt    
    return y1.reshape(-1)

def stoermer_verlet(rhs, y0, T, N):                          # WICHTIG y0 = [x0, y0] nicht [y0, y_dot0] !!!!!!!
    t, dt = np.linspace(0.0, T, N + 1, retstep=True)

    y = np.empty((N + 1,) + y0.shape)
    y[0, ...] = y0

# einschritt: 
    v_temp =  y0[1] + 0.5 * dt * rhs(dt, y0)                  # v(0)   = v0 + 0.5 * a0 * t

    for n in range(0, N): 
        y[n + 1, :] = y[n, :] + dt * v_temp                   # y(n+1) = y(n)  + v(n)   * dt 
        v_temp      = v_temp  + dt * rhs(dt, y[n + 1, :])     # v(n+1) = v(n)  + v(n-1) * dt                                          
            
# zweischritt: 
    y[1, :] = y0 + dt * y0[1] + 0.5 * dt ** 2 * rhs(dt, y0)    #    y1 = y0 + v0 * dt + 0.5 * a0 * dt**2 

    for k in range(1, N):
        y[k + 1, :] = -y[k - 1, :] + 2.0 * y[k, :] + dt**2 * rhs(dt, y[k, :])    # y(k+1) = -y(k-1) + 2*y(k) + dt**2 * a

    return t, y          


In [ ]:
# Splitting:

def splitting_parameters(method):
    if method == "LT":
        s = 1 
        a = np.zeros(s)
        b = np.zeros(s)
        a[0] = 1.0
        b[0] = 1.0
    elif method == "SS":
        s = 2
        a = np.zeros(s)
        b = np.zeros(s)
        a[0] = 0.5
        a[1] = 0.5
        b[0] = 1.0
    return a, b

def rhs(q, p):
    return 420*q + 69*p

def Phi_A(y, dt):
    q, p = y[0], y[1]  # y = [q, p]
    return np.array([q + dt * p, p])  

def Phi_B(y, dt):
    q, p = y[0], y[1]  
    return np.array([q, p + dt * rhs(q, p)]) 

def splitting_method_step(y, dt, method = "SS"):
    a, b = splitting_parameters(method)   
    for ai, bi in zip(a, b):
        y = Phi_B(Phi_A(y, ai * dt), bi * dt)
    return y

def splitting_method(y0, t_end, N):
    return integrate(splitting_method_step, y0, t_end, N) 

In [ ]:
import numpy as np
A = 1; B = 1; epsilon = 1e-6; omega = 1e-3; dt=0.1
y = [0.5, 0.1]

# Exam FS19 / Serie 8
# rhs = [dθ/dt = ω, dω/dt =  - c / (m * L * L) * θ + g/L * sin(θ)] 
rhs   = [y[1], A*y[0] + B*np.sin(y[0])]

#       [ω, 0], update θ: dθ/dt = ω    # THIS NEED THE +dt*...
P_a   = [y[1], 0.0]
#       [0, A * θ + B * sin(θ)], update ω: dω/dt = A θ + B sin(θ)
P_b   = [0.0 , A * y[0] + B * np.sin(y[0])] 

#       [θ + dt * ω, ω]
Phi_A = [y[0] + dt * y[1], y[1]]
#       [θ, ω + dt * (A * θ + B * sin(θ))] 
Phi_B = [y[0], y[1] + dt * (A * y[0] + B * np.sin(y[0]))]

#       [θ + dt * ω, ω + dt * A * θ]
Phi_A = [y[0] + dt * y[1], y[1] + dt * A * y[0]]
#       [θ, ω + dt * B * sin(θ)]
Phi_B = [y[0], y[1] + dt * B * np.sin(y[0])]


# Exam FS18
# rhs = [du/dt = p, dp/dt = - q + (epsilon*omega) * cos(omega * dt)]
# P_a = [q + p * dt, p], update u: du/dt = p
# P_b = [q, p - u * dt + (epsilon*omega) * cos(omega * dt) * dt]
rhs   = np.array([y[1], -y[0] + (epsilon*omega) * np.cos(omega * dt)])
Phi_a = np.array([y[0] + y[1] * dt , y[1]])
Phi_b = np.array([y[0]             , y[1] + dt * (-y[0] + (epsilon*omega) * np.cos(omega * dt) )])

# Serie 7: graviatation
Phi_a = np.array([y[0] + dt * gradT(y[1]), y[1]]) 
Phi_b = np.array([y[0], y[1] - dt * gradV(y[0])])       



def splitting_ode(y, A, B, dt): # Page: 166
    # rhs = [dθ/dt = ω, dω/dt =  - c / (m * L * L) * θ + g/L * sin(θ)] 
    rhs   = [y[1], A*y[0] + B*np.sin(y[0])]
    #       [θ + dt * ω, ω]
    Phi_A = [y[0] + dt * y[1], y[1]]
    #       [θ, ω + dt * (A * θ + B * sin(θ))] 
    Phi_B = [y[0]            , y[1] + dt * (A * y[0] + B * np.sin(y[0]))]

def splitting_ode(rhs, q, p, dt):
    Phi_A = [q + dt * p, p]
    Phi_B = [q         , p + dt * rhs(q, p)]
    return Phi_B(Phi_A(q, p))

# even more general splitting:
def splitting_ode(rhs, y, dt):
    return [y[0] + dt * y[1], y[1] + dt * rhs(y[0], y[1])]

In [ ]:
# EXAM CODE: 

# FS20: 
# Butcher-Tabelle der Kutta38-Regel:
import matplotlib.pyplot as plt
import numpy as np
from sympy import symbols, Matrix, eye, lambdify, nsimplify

def stability_function():
    z = symbols("z")
    U = Matrix([[0,0,0,0],[1/2,0,0,0],[1/4,1/4,0,0],[0,-1,2,0]])
    b = Matrix([1/6, 0, 4/6, 1/6]).T
    S_ = sum(b * (eye(4) - z*U).inv())
    S = 1 + z * S_

    print(nsimplify(S))
    return lambdify(z, S) 

def basic_plot(S):
    x = np.linspace(-5, 5, 1000)
    X, Y = np.meshgrid(x, x)
    z = X + 1j*Y
    Z = np.abs(S(z)) < 1

    plt.contourf(X, Y, Z)
    plt.grid()

def compute_h(S, lam = -3 - 3j):
    for h in np.linspace(0.001, 1, 1000):
        z = h * lam
        if abs(S(z)) > 1:
            print(f"Converges for h ≈ {h:.5f} (|S(z)| = {abs(S(z)):.5f})")
            break

In [ ]:
# ADDITIONAL CODE 

def rk_step(f, y0, t0, dt):  # ONLY for explicit 
    k1 = f(t0 + dt * c[0], y0)
    k2 = f(t0 + dt * c[1], y0 + dt *(A[1,0] * k1))
    k3 = f(t0 + dt * c[2], y0 + dt *(A[2,0] * k1 + A[2,1] * k2))
    k4 = f(t0 + dt * c[3], y0 + dt *(A[3,0] * k1 + A[3,1] * k2 + A[4,1] * k3))
    y1 = y0 + dt * (b[0] * k1 + b[1] * k2 + b[2] * k3 + b[3] * k4)
    return y1

def expEV_step(h, yn, f, Df): # Take a look 
    return yn + (expm(h * Df(yn) ) - np.identity(len(Df(yn)))) @ solve(Df(yn) , f(yn))    # y_k+1     = y_k     + h * phi(h J_f) * f(y_k)

def row_step(f, Jf, y0, dt):
    n = y0.shape[0]
    a = 1.0 / (2.0 + np.sqrt(2.0))
    d31 = - (4.0 + np.sqrt(2.0)) / (2.0 + np.sqrt(2.0))
    d32 =   (6.0 + np.sqrt(2.0)) / (2.0 + np.sqrt(2.0))

    I = np.identity(n)
    J = Jf(y0)
    A = I - a*dt*J

# row 2:
    b1 = f(y0)       ;   b2 = f(y0+0.5*dt*k1) - a*dt* J @ k1
    k1 = solve(A, b1);   k2 = solve(A, b2)
    y1 = y0 + dt*k2

# row 3:
    b1 = f(y0)       ;   b2 = f(y0+0.5*dt*k1) - a*dt* J @ k1;   b3 = f(y0+dt*k2) - d31*dt* J @ k1 - d32*dt* J @ k2
    k1 = solve(A, b1);   k2 = solve(A, b2)                    ;   k3 = solve(A, b3)
    y1 = y0 + dt/6.0*(k1 + 4.0*k2 + k3)

    return y1

def nbody_pdot(t, q, m, G, n_bodies):
    dpdt = np.empty_like(q)

    for k in range(n_bodies):
        dq   = q - q[k, :]
        mimk = m * m[k,:]
       
        norm3 = np.linalg.norm(dq, axis=1) ** 3
        norm3 = norm3.reshape((-1,1))

        dpdt[k, :] = G * sum(mimk * dq / (norm3 + 1e-200), axis=0)
    
    return dpdt

## Chapter 5: Nullstellensuche

In [ ]:
def bisect(x_low, x_high, f, maxiter, n_iter=0, r_tol=1e-20):  # Page: 247 

    x_mid = (x_low + x_high) / 2
    
    if x_high - x_low < r_tol or n_iter > maxiter:
        return x_mid

    if f(x_low) * f(x_mid) < 0:
        return bisect(x_low, x_mid,  f, maxiter, n_iter + 1)
    else:
        return bisect(x_mid, x_high, f, maxiter, n_iter + 1)

def secant(f, x0, x1, atol, rtol, maxit): # Page 251
    x = [x0, x1]
    tol = atol + rtol * norm(x)

    for iter in range(maxit):
        diff_f = (f(x[-1]) - f(x[-2])) / (x[-1] - x[-2])
        x_new = x[-1] - f(x[-1]) / diff_f

        x.append(x_new) 
        n_iters = iter + 1

        if abs(x[-1] - x[-2]) < tol:
            break
    
    return x[-1], n_iters, x


def newton(F, DF, x0, atol, rtol, maxiter): # Page 253
    x = np.atleast_2d(x0) 
    x_seq = [x0]
    tol = atol + rtol * norm(x)

    for iter in range(maxiter):
        s = np.linalg.solve(DF(x), F(x))
        x = x - s
        x_seq.append(x)
        n_iters = iter + 1

        if norm(s) < tol:
            break
    
    return x, n_iters, x_seq

def find_jacobian():  # Page 339
    x, y = sp.symbols('x y')
    
    F_mat = sp.Matrix([sp.sin(sp.pi * sp.sqrt(x**2 + 4*y**2)),
                     x**2 + y - 4 * sp.sin(sp.pi * x)**2 + 1])
    J_mat = F_mat.jacobian([x, y])

    F_ = sp.lambdify((x,y), F_mat, 'numpy')
    J_ = sp.lambdify((x,y), J_mat, 'numpy')

    F = lambda w: np.array(F_(*w), dtype=float).flatten()
    J = lambda w: np.array(J_(*w), dtype=float).reshape(2, 2)
    def F(w):
        return(F_(*w).ravel())
    def J(w):
        return(J_(*w))
    return F_mat, J_mat

def compute_errors_and_order(x0, F, DF): # Page 238 only theory for order 
    _, n_iter, x_seq = newton(F, DF, x0, 1.0e-7, 1.0e-12, 10 ** 3)

    exact  = fsolve(F, x0)
    errors = [norm(exact - x) for x in x_seq]

    for i in range(1, n_iter - 2):    # there are only n_inter -2 available solutions        
        order = (np.log(errors[i + 1] + 1e-21) - np.log(errors[i]     + 1e-21)) / (
                (np.log(errors[i]     + 1e-21) - np.log(errors[i - 1] + 1e-21)))   
    
    return x_seq[1:], errors[:-1], order 


## Chapter 6: Integration für steife Differentialgleichungen

## Chapter 7: Intermezzo Lineare Algebra

## Chapter 8: Ausgleichsrechnung

In [ ]:
""" Linear Regression """
def compute_interpolation(x_gemessen, y_gemessen, degree):
    A = np.vander(x_gemessen, N=degree+1, increasing=True)
    b = y_gemessen
    a0, a1 = lstsq(A, b, rcond=None)[0] 
    
    x_plot = np.linspace(min(x_gemessen), max(x_gemessen), 100)
    y_plot = a1 * x_plot + a0
    return x_plot, y_plot

""" Lineares gleichungssystem loesen """

# c  =  R^-1 * Q.T * b  =  Vh.T * S * U.T * b  =  lstsq(A,b)[0] 
def compute_interpolation(x, y, degree, method):
    A = np.vander(x, N=degree+1, increasing=True)
    b = y 

    if method == "qr": # Page: 299 and 324
        Q, R = qr(A) # No mode="economic"
        c    = solve(R, Q.T @ b)

    if method == "svd": # Page: 305 and 323
        U, sigmas, Vh = svd(A) # No full_matrices=False 
        rA = matrix_rank(A)
        c = Vh[:rA, :].T  @  np.diag(1/sigmas[:rA]) @  U[:, :rA].T  @ b
        # A = U @ np.diag(sigmas) @ Vh

    if method == "psuedo inverse":
        c = pinv(A) @ b

    if method == "lstsq":  
        c = lstsq(A, b, rcond=None)[0]

    coeffs = c[::-1] 
    x_plot = np.linspace(min(x), max(x), N=100)
    y = np.polyval(coeffs, x_plot)  
    return coeffs, y

""" Least Squares mit linearen Nebenbedingungen """ # Kann sein dass es nicht kommt!!
def lstsqlincon(A, b, C, d):    
    zero = np.zeros((C.shape[0], C.shape[0]))
    K    = np.block([[A.T@A, C.T],
                     [C,    zero]])
    rhs  = np.concatenate([A.T@b, d])
    # sol  = np.linalg.solve(K, rhs) # only if K is not ill defined 
    sol  = np.linalg.lstsq(K, rhs, rcond=None)[0]
    m, n = A.shape
    return sol[:n]

def lstsqlincon2(A, b, C, d):   
    rC = matrix_rank(C)
    U, S, Vh = np.linalg.svd(C)
    x0 = Vh[:rC, :].T  @  np.diag(1/S[:rC]) @  U[:, :rC].T  @ d
    return x0 + Vh[rC:, :].T @ lstsq(A @ Vh[rC:, :].T, b  - A @ x0, rcond=None)[0]